# Задача регрессии для CC50
В этом разделе мы будем предсказывать показатель цитотоксичности CC50, используя аналогичный подход: логарифмирование целевой переменной (pCC50) и оптимизацию гиперпараметров.

In [ ]:
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import seaborn as sns
import lightgbm as lgb


In [ ]:


# Load the dataset
file_path = '/content/cleaned_molecular_data.csv'
df = pd.read_csv(file_path)

# Display basic information
print("Dataset Shape:", df.shape)
display(df.head())
print(df.info())

Dataset Shape: (966, 214)


,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,is_outlier
0,6.239374,175.482382,28.125000,5.094096,5.094096,0.387225,0.387225,0.417362,42.928571,384.652,...,0,0,0,0,0,0,0,3,0,1
1,0.771831,5.402819,7.000000,3.961417,3.961417,0.533868,0.533868,0.462473,45.214286,388.684,...,0,0,0,0,0,0,0,3,0,1
2,223.808778,161.142320,0.720000,2.627117,2.627117,0.543231,0.543231,0.260923,42.187500,446.808,...,0,0,0,0,0,0,0,3,0,1
3,1.705624,107.855654,63.235294,5.097360,5.097360,0.390603,0.390603,0.377846,41.862069,398.679,...,0,0,0,0,0,0,0,4,0,1
4,107.131532,139.270991,1.300000,5.150510,5.150510,0.270476,0.270476,0.429038,36.514286,466.713,...,0,0,0,0,0,0,0,0,0,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Columns: 214 entries, IC50, mM to is_outlier
dtypes: float64(107), int64(107)
memory usage: 1.6 MB
None


In [ ]:
# 1. Подготовка данных для CC50
target_cc50 = 'CC50, mM'
exclude_cols_cc = ['IC50, mM', 'CC50, mM', 'SI', 'IC50_above_median', 'CC50_above_median', 'SI_above_median', 'SI_above_8', 'is_outlier']

# Признаки (numeric)
relevant_features_cc = [col for col in df.select_dtypes(include=[np.number]).columns if col not in exclude_cols_cc]

# Убираем строки, где CC50 может быть NaN или некорректным (если есть)
df_cc = df.dropna(subset=[target_cc50]).copy()

X_cc = df_cc[relevant_features_cc]
y_cc_log = -np.log10(df_cc[target_cc50])

# Разделение на выборки
X_train_cc, X_test_cc, y_train_cc, y_test_cc = train_test_split(X_cc, y_cc_log, test_size=0.2, random_state=42)

print(f"Samples for CC50 task: {len(df_cc)}")
print(f"Target distribution (pCC50) mean: {y_cc_log.mean():.2f}")

Samples for CC50 task: 966
Target distribution (pCC50) mean: -2.41


### Базовые модели (стандартные параметры) для CC50

In [ ]:
# Определение стандартных моделей для baseline
standard_models = {
    "GBR (Default)": GradientBoostingRegressor(random_state=42),
    "RF (Default)": RandomForestRegressor(random_state=42),
    "LGBM (Default)": lgb.LGBMRegressor(random_state=42, verbosity=-1)
}

baseline_cc50 = []
for name, model in standard_models.items():
    model.fit(X_train_cc, y_train_cc)
    p_tr= model.predict(X_train_cc)
    p_te= model.predict(X_test_cc)
    baseline_cc50.append({
        "Model": name,
        "Train RMSE": np.sqrt(mean_squared_error(y_train_cc, p_tr)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test_cc, p_te)),
        "Test R2": r2_score(y_test_cc, p_te)
    })
display(pd.DataFrame(baseline_cc50))

,Model,Train RMSE,Test RMSE,Test R2
0,GBR (Default),0.384005,0.568525,0.452285
1,RF (Default),0.326969,0.547672,0.491729
2,LGBM (Default),0.297305,0.569912,0.449610


### Полный цикл регрессии для CC50
Сравним GBR, RF и LGBM, подберем гиперпараметры и проверим влияние фильтрации выбросов на предсказание.

In [ ]:
def run_optuna_cc50(model_class, n_trials=150):
    def objective(trial):
        if model_class == GradientBoostingRegressor:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 10, 400),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
                'max_depth': trial.suggest_int('max_depth', 1, 7),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'random_state': 42
            }
        elif model_class == RandomForestRegressor:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 10, 400),
                'max_depth': trial.suggest_int('max_depth', 1, 15),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
                'random_state': 42
            }
        else: # LGBM
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 10, 400),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 2, 60),
                'verbosity': -1,
                'random_state': 42
            }

        model = model_class(**params)

        # Обучаем на тренировочных данных
        model.fit(X_train_cc, y_train_cc)

        # Предсказания для замера качества
        train_p = model.predict(X_train_cc)
        test_p = model.predict(X_test_cc)

        train_rmse = np.sqrt(mean_squared_error(y_train_cc, train_p))
        test_rmse = np.sqrt(mean_squared_error(y_test_cc, test_p))

        # Минимизируем тренировочное RMSE + штраф за разницу с тестом
        # Коэффициент штрафа можно регулировать
        gap = abs(train_rmse - test_rmse)
        penalty_factor = 0.5

        return train_rmse + penalty_factor * gap

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

print("Tuning GBR for CC50 (Test-benchmarked)... ")
best_gbr_cc = run_optuna_cc50(GradientBoostingRegressor)

print("Tuning RF for CC50 (Test-benchmarked)... ")
best_rf_cc = run_optuna_cc50(RandomForestRegressor)

print("Tuning LGBM for CC50 (Test-benchmarked)... ")
best_lgbm_cc = run_optuna_cc50(lgb.LGBMRegressor)

[I 2026-05-23 08:51:07,275] A new study created in memory with name: no-name-a392240e-9c30-4edc-835a-b06699cef1b5


Tuning GBR for CC50 (Test-benchmarked)... 


[I 2026-05-23 08:51:28,088] Trial 0 finished with value: 0.45349779309532334 and parameters: {'n_estimators': 365, 'learning_rate': 0.00500187059345782, 'max_depth': 7, 'subsample': 0.8807209984038791}. Best is trial 0 with value: 0.45349779309532334.
[I 2026-05-23 08:51:37,210] Trial 1 finished with value: 0.4292993555336308 and parameters: {'n_estimators': 206, 'learning_rate': 0.034004558865460686, 'max_depth': 7, 'subsample': 0.69157290973724}. Best is trial 1 with value: 0.4292993555336308.
[I 2026-05-23 08:51:58,445] Trial 2 finished with value: 0.4255735726453129 and parameters: {'n_estimators': 365, 'learning_rate': 0.013722275083231355, 'max_depth': 7, 'subsample': 0.8879610281204171}. Best is trial 2 with value: 0.4255735726453129.
[I 2026-05-23 08:52:14,536] Trial 3 finished with value: 0.4324419295322053 and parameters: {'n_estimators': 297, 'learning_rate': 0.028189134625473178, 'max_depth': 6, 'subsample': 0.9896138874966188}. Best is trial 2 with value: 0.425573572645312

Tuning RF for CC50 (Test-benchmarked)... 


[I 2026-05-23 09:30:17,982] Trial 0 finished with value: 0.6246211254104586 and parameters: {'n_estimators': 396, 'max_depth': 2, 'max_features': None}. Best is trial 0 with value: 0.6246211254104586.
[I 2026-05-23 09:30:18,828] Trial 1 finished with value: 0.6133210942057192 and parameters: {'n_estimators': 303, 'max_depth': 3, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.6133210942057192.
[I 2026-05-23 09:30:20,106] Trial 2 finished with value: 0.686602557762939 and parameters: {'n_estimators': 176, 'max_depth': 1, 'max_features': None}. Best is trial 1 with value: 0.6133210942057192.
[I 2026-05-23 09:30:20,436] Trial 3 finished with value: 0.43930141157937175 and parameters: {'n_estimators': 49, 'max_depth': 14, 'max_features': 'sqrt'}. Best is trial 3 with value: 0.43930141157937175.
[I 2026-05-23 09:30:32,563] Trial 4 finished with value: 0.440166872356096 and parameters: {'n_estimators': 174, 'max_depth': 15, 'max_features': None}. Best is trial 3 with value: 0.43930141

Tuning LGBM for CC50 (Test-benchmarked)... 


[I 2026-05-23 09:35:30,498] Trial 0 finished with value: 0.4405310415761464 and parameters: {'n_estimators': 296, 'learning_rate': 0.022288142476887693, 'num_leaves': 26}. Best is trial 0 with value: 0.4405310415761464.
[I 2026-05-23 09:35:31,336] Trial 1 finished with value: 0.44552533252578475 and parameters: {'n_estimators': 164, 'learning_rate': 0.031135511833347635, 'num_leaves': 34}. Best is trial 0 with value: 0.4405310415761464.
[I 2026-05-23 09:35:31,890] Trial 2 finished with value: 0.44736032069693943 and parameters: {'n_estimators': 268, 'learning_rate': 0.07285173599254513, 'num_leaves': 9}. Best is trial 0 with value: 0.4405310415761464.
[I 2026-05-23 09:35:32,362] Trial 3 finished with value: 0.4539095383073938 and parameters: {'n_estimators': 108, 'learning_rate': 0.06558521647868726, 'num_leaves': 17}. Best is trial 0 with value: 0.4405310415761464.
[I 2026-05-23 09:35:34,215] Trial 4 finished with value: 0.43272581163373547 and parameters: {'n_estimators': 335, 'learn

In [ ]:
best_gbr_cc

{'n_estimators': 338,
 'learning_rate': 0.07917569954940945,
 'max_depth': 7,
 'subsample': 0.8950529393998701}

In [ ]:
best_rf_cc

{'n_estimators': 239, 'max_depth': 15, 'max_features': 'log2'}

In [ ]:
best_lgbm_cc

{'n_estimators': 331, 'learning_rate': 0.07439435321346234, 'num_leaves': 42}

In [ ]:
best_lgbm_cc={'n_estimators': 331, 'learning_rate': 0.07439435321346234, 'num_leaves': 42}
best_rf_cc={'n_estimators': 239, 'max_depth': 15, 'max_features': 'log2'}
best_gbr_cc={'n_estimators': 338,
 'learning_rate': 0.07917569954940945,
 'max_depth': 7,
 'subsample': 0.8950529393998701}

In [ ]:
# Сравнение оптимизированных моделей на CC50
cc_results = []
models_cc = {
    "GBR": GradientBoostingRegressor(**best_gbr_cc, random_state=42),
    "RF": RandomForestRegressor(**best_rf_cc, random_state=42),
    "LGBM": lgb.LGBMRegressor(**best_lgbm_cc, random_state=42, verbosity=-1)
}

for name, model in models_cc.items():
    model.fit(X_train_cc, y_train_cc)
    tr_p= model.predict(X_train_cc)
    te_p = model.predict(X_test_cc)
    cc_results.append({
        "Model": name,
        "Train RMSE": np.sqrt(mean_squared_error(y_train_cc, tr_p)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test_cc, te_p)),
        "Test R2": r2_score(y_test_cc, te_p)
    })

display(pd.DataFrame(cc_results))

,Model,Train RMSE,Test RMSE,Test R2
0,GBR,0.287550,0.549681,0.487992
1,RF,0.328339,0.542752,0.500819
2,LGBM,0.286731,0.567318,0.454610


удалосm немного улучшить метрику качетсва на тесте

### CC50: Оценка на данных без выбросов
Теперь проверим, улучшится ли предсказание CC50, если мы уберем выбросы (те же 49 молекул, что и для IC50).

In [ ]:
# Фильтруем те же данные для CC50
df_cc_f = df_cc[df_cc['is_outlier'] == 1].copy()
X_cc_f = df_cc_f[relevant_features_cc]
y_cc_f = -np.log10(df_cc_f[target_cc50])

X_tr_cc_f, X_te_cc_f, y_tr_cc_f, y_te_cc_f = train_test_split(X_cc_f, y_cc_f, test_size=0.2, random_state=42)

# Проверяем лучшую модель (например, RF) на чистых данных
final_model_cc = RandomForestRegressor(**best_rf_cc, random_state=42)
final_model_cc.fit(X_tr_cc_f, y_tr_cc_f)

f_preds = final_model_cc.predict(X_te_cc_f)
print(f"CC50 Results (Outliers Removed):")
print(f"RMSE: {np.sqrt(mean_squared_error(y_te_cc_f, f_preds)):.4f}")
print(f"R2:   {r2_score(y_te_cc_f, f_preds):.4f}")

CC50 Results (Outliers Removed):
RMSE: 0.5641
R2:   0.4066


интерсно что в данной задаче удаление выбросов не улучшает модель возможно предельные значения имеют взаимосвясь с cc50